In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from mplsoccer import Pitch, VerticalPitch

pitch = Pitch(pitch_type='statsbomb', pitch_color='green', corner_arcs=True, line_color='white', positional=True)

In [ ]:
df_stories = pd.read_parquet('/Path')

In [ ]:
mask_hammarby = (df_stories['team_name'] == 'Hammarby') & (df_stories['type'] == 'pass') & (df_stories['throw_in'] == False) & (df_stories['corner'] == False) & (df_stories['goal_kick'] == False) & (df_stories['free_kick'] == False)
df_pass = df_stories.loc[mask_hammarby, ['player_name', 'player_recipient_name', 'player_id', 'pass_recipient_id', 'start_x', 'start_y', 'end_x', 'end_y']]

df_pass['player_name'] = df_pass['player_name'].apply(lambda x: str(x).split()[-1])
df_pass['player_recipient_name'] = df_pass['player_recipient_name'].apply(lambda x: str(x).split()[-1])

In [ ]:
df_pass['start_x'] *= 1.2
df_pass['end_x']   *= 1.2
df_pass['start_y'] *= 0.8
df_pass['end_y']   *= 0.8

In [ ]:
df_pass.to_csv('Path', index=False)

In [ ]:
scatter_df = pd.DataFrame(columns=['player_name', 'start_x', 'start_y', 'no'])
for i, name in enumerate(df_pass['player_name'].unique()):
    passx = df_pass.loc[df_pass['player_name'] == name]['start_x'].to_numpy()
    recx = df_pass.loc[df_pass['player_recipient_name'] == name]['end_x'].to_numpy()
    passy = df_pass.loc[df_pass['player_name'] == name]['start_y'].to_numpy()
    recy = df_pass.loc[df_pass['player_recipient_name'] == name]['end_y'].to_numpy()
    scatter_df.at[i, 'player_name'] = name

    scatter_df.at[i, 'start_x'] = np.mean(np.concatenate([passx, recx]))
    scatter_df.at[i, 'start_y'] = np.mean(np.concatenate([passy, recy]))

    scatter_df.at[i, 'no'] = df_pass.loc[df_pass['player_name'] == name].count().iloc[0]

scatter_df['marker_size'] = (scatter_df['no'] / scatter_df['no'].max() * 1500)

for col in ['start_x', 'start_y', 'marker_size', 'no']:
    scatter_df[col] = pd.to_numeric(scatter_df[col])

scatter_df = scatter_df.nlargest(15, 'no').copy()

In [ ]:
df_pass['pair_key'] = df_pass.apply(lambda x: '_'.join(sorted([x['player_name'], x['player_recipient_name']])), axis=1)
lines_df = (df_pass.groupby('pair_key').size().reset_index(name='pass_count'))
lines_df = lines_df[lines_df['pass_count'] > 2]

In [ ]:
top_players = set(scatter_df['player_name'])

lines_df = lines_df.copy()
lines_df['p1'] = lines_df['pair_key'].str.split('_').str[0]
lines_df['p2'] = lines_df['pair_key'].str.split('_').str[1]

lines_df = lines_df[lines_df['p1'].isin(top_players) & lines_df['p2'].isin(top_players)]

In [ ]:
fig, ax = pitch.grid(grid_height=0.9, title_height=0.06, axis=False,
                     endnote_height=0.04, title_space=0, endnote_space=0)
axp = ax['pitch']

pitch.scatter(scatter_df['start_x'], scatter_df['start_y'],
              s=scatter_df['marker_size'], color='#ffcd00', edgecolors='grey',
              linewidth=1, alpha=1, ax=axp, zorder=3)

xy = scatter_df[['start_x', 'start_y']].to_numpy(float)
names = scatter_df['player_name'].astype(str).to_list()
n = len(names)

diff = xy[:, None, :] - xy[None, :, :]
dist = np.sqrt((diff**2).sum(axis=2))
dist = dist + np.eye(n) * 1e9

nn = dist.argmin(axis=1)           
vec = xy - xy[nn]                
norm = np.linalg.norm(vec, axis=1, keepdims=True)
unit = np.divide(vec, norm, out=np.zeros_like(vec), where=norm > 0)

dmin = dist.min(axis=1)
offset_mag = np.where(dmin < 4, 5.0, np.where(dmin < 8, 3.5, 2.5))
offset = unit * offset_mag[:, None]

for i, name in enumerate(names):
    x, y = xy[i]
    dx, dy = offset[i]
    tx, ty = x + dx, y + dy

    tx = np.clip(tx, 0, 120)
    ty = np.clip(ty, 0, 80)

    ha = 'left' if dx >= 0 else 'right'

    pitch.annotate(
        name, xy=(tx, ty),
        c='black', va='center', ha=ha, weight='bold', size=12,
        ax=axp, zorder=6,
        path_effects=[pe.withStroke(linewidth=4, foreground='white')]
    )

# fig.suptitle('Top 15 players with most passes in Hammarby 2024', fontsize=22)
plt.show()

In [ ]:
fig, ax = pitch.grid(grid_height=0.9, title_height=0.06, axis=False,
                     endnote_height=0.04, title_space=0, endnote_space=0)
axp = ax['pitch']

pos = scatter_df.set_index('player_name')[['start_x','start_y']].to_dict('index')

max_pass = lines_df['pass_count'].max() if len(lines_df) else 1

tmp = lines_df.copy()
tmp['p1'] = tmp['pair_key'].str.split('_').str[0]
tmp['p2'] = tmp['pair_key'].str.split('_').str[1]

for _, row in tmp.iterrows():
    p1, p2 = row['p1'], row['p2']
    if (p1 not in pos) or (p2 not in pos):
        continue

    x1, y1 = pos[p1]['start_x'], pos[p1]['start_y']
    x2, y2 = pos[p2]['start_x'], pos[p2]['start_y']

    lw = (row['pass_count'] / max_pass) * 10
    pitch.lines(x1, y1, x2, y2, alpha=0.35, lw=lw, zorder=2, color='#ffcd00', ax=axp)

pitch.scatter(scatter_df['start_x'], scatter_df['start_y'],
              s=scatter_df['marker_size'], color='#ffcd00', edgecolors='grey',
              linewidth=1, alpha=1, ax=axp, zorder=3)

xy = scatter_df[['start_x','start_y']].to_numpy(float)
names = scatter_df['player_name'].astype(str).to_list()

diff = xy[:, None, :] - xy[None, :, :]
dist = np.sqrt((diff**2).sum(axis=2))
dist = dist + np.eye(n) * 1e9

nn = dist.argmin(axis=1)
vec = xy - xy[nn]
norm = np.linalg.norm(vec, axis=1, keepdims=True)
unit = np.divide(vec, norm, out=np.zeros_like(vec), where=norm > 0)

dmin = dist.min(axis=1)

offset_mag = np.where(dmin < 4, 5.0, np.where(dmin < 8, 3.5, 2.5))
offset = unit * offset_mag[:, None]

for i, name in enumerate(names):
    x, y = xy[i]
    dx, dy = offset[i]
    tx, ty = x + dx, y + dy

    tx = np.clip(tx, 0, 120)
    ty = np.clip(ty, 0, 80)

    ha = 'left' if dx >= 0 else 'right'

    pitch.annotate(name, xy=(tx, ty),
                    c='black', va='center', ha=ha, weight='bold', size=16,
                    ax=axp, zorder=6,
                    path_effects=[pe.withStroke(linewidth=4, foreground='white')])

# fig.suptitle('Hammarby Passing Network in Allsvanskan 2024', fontsize=30)
plt.show()